In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime
import logging
import sys
from PIL import Image
import json
from shapely.geometry import box

# Set up project path for Jupyter notebook
project_root = Path().resolve().parent  # Assuming notebook is in tests directory
sys.path.append('')  # Use your actual path

# Set up logging
logging.basicConfig(level=logging.INFO)

# Import SecureImageEncoder
from src.core.security.secure_encoding import SecureImageEncoder, SecureAPILayer

def load_test_image(image_path):
    """Load and preprocess test image"""
    img = Image.open(image_path)
    return np.array(img)

def visualize_results(original, decoded, title="Image Comparison"):
    """Visualize original and decoded images"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    
    # Original image
    ax1.imshow(original)
    ax1.set_title('Original Image')
    ax1.axis('off')
    
    # Decoded image
    ax2.imshow(decoded)
    ax2.set_title('Decoded Image')
    ax2.axis('off')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Set up paths for Jupyter
notebook_dir = Path().resolve()
image_path = notebook_dir / "data" / "a1.png"
output_dir = notebook_dir / "output"
output_dir.mkdir(exist_ok=True)

print(f"Loading image from: {image_path}")

# Load test image
try:
    image_data = load_test_image(image_path)
    print(f"Image loaded successfully. Shape: {image_data.shape}")
except Exception as e:
    print(f"Error loading image: {e}")
    raise

# Create test metadata
metadata = {
    "timestamp": datetime.now().isoformat(),
    "source": "test_image",
    "resolution": image_data.shape,
    "format": "PNG"
}

# Initialize secure encoder with a test key
master_key = "test_secure_key_2024"
encoder = SecureImageEncoder(master_key)

# Test basic encoding/decoding
print("\nTesting basic encoding/decoding...")
try:
    # Encode
    encrypted_data, secure_metadata = encoder.encode_image(
        image_data,
        metadata,
        protection_level='high'
    )
    print(f"Image encoded successfully. Encrypted size: {len(encrypted_data)} bytes")
    
    # Generate access token
    access_token = secure_metadata['access_token']
    
    # Decode
    decoded_image = encoder.decode_image(
        encrypted_data,
        secure_metadata,
        access_token
    )
    
    if decoded_image is not None:
        print("Image decoded successfully")
        visualize_results(image_data, decoded_image, "Basic Encoding Test")
    else:
        print("Error: Failed to decode image")
        
except Exception as e:
    print(f"Error in basic encoding test: {e}")

# Test geo-privacy encoding
print("\nTesting geo-privacy encoding...")
try:
    # Create sample geometry (bounding box)
    bounds = box(0, 0, 100, 100)
    
    # Encode with geo-privacy
    encrypted_data, secure_metadata = encoder.encode_with_geo_privacy(
        image_data,
        bounds,
        metadata,
        protection_level='high',
        layout_type='grid'
    )
    print(f"Image encoded with geo-privacy. Encrypted size: {len(encrypted_data)} bytes")
    
    # Decode with geo-privacy
    decoded_image, decoded_geom = encoder.decode_with_geo_privacy(
        encrypted_data,
        secure_metadata,
        secure_metadata['access_token']
    )
    
    if decoded_image is not None:
        print("Image and geometry decoded successfully")
        visualize_results(image_data, decoded_image, "Geo-Privacy Encoding Test")
    else:
        print("Error: Failed to decode image with geo-privacy")
        
except Exception as e:
    print(f"Error in geo-privacy test: {e}")

# Test API layer
print("\nTesting Secure API Layer...")
try:
    api_layer = SecureAPILayer(master_key)
    
    # Register test API key
    test_api_key = "test_api_key_2024"
    api_permissions = {
        "read": True,
        "write": True,
        "geo_privacy": True
    }
    api_layer.register_api_key(test_api_key, api_permissions)
    
    # Test tile encoding through API layer
    encrypted_tile, tile_metadata = api_layer.encode_tile(
        image_data,
        metadata,
        protection_level='high'
    )
    print("Tile encoded through API layer successfully")
    
    # Test tile decoding
    decoded_tile = api_layer.decode_tile(
        encrypted_tile,
        tile_metadata,
        tile_metadata['access_token']
    )
    
    if decoded_tile is not None:
        print("Tile decoded through API layer successfully")
        visualize_results(image_data, decoded_tile, "API Layer Test")
    else:
        print("Error: Failed to decode tile through API layer")
        
except Exception as e:
    print(f"Error in API layer test: {e}")

print("\nTesting complete!")

In [ ]:
main()